# Kernel-based MC Neutron Transport


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from nuclear_data import *
from mf6 import *
from mf4 import *
from transport import *
from transport_kernel import *
from decay_activity import *
from notebook_helpers import *

#PROJECT_ROOT = Path.cwd()
CURRENT_ROOT = Path.cwd()
PROJECT_ROOT = Path("/Users/mathewoaks/Desktop/Activation_Code/neutron_activation/Activation_Data/nuclear_data_npz")

#print("Project folder:", PROJECT_ROOT)

## 0. Folders and Directories

In [ ]:
# ============================================================
# Project folders
# ============================================================
# This notebook expects this repo layout:
#
# Kernel_Based_MC_NT/
#     nuclear_data.py
#     transport.py
#     mf4.py
#     mf6.py
#     decay_activity.py
#     notebook_helpers.py
#     materials/
#         Steel.py
#         Concrete.py
#     data/
#         neutron_npz/
#         decay_npz/
#     kernels/
#     outputs/
#
# The data folder is intentionally not included on GitHub.
# Users must provide their own processed NPZ nuclear data.
# ============================================================

#DATA_DIR = CURRENT_ROOT / "data"
#XS_DIR = DATA_DIR / "neutron_npz"
#DECAY_DIR = DATA_DIR / "decay_npz"
XS_DIR = PROJECT_ROOT / "neutron_npz"
DECAY_DIR = PROJECT_ROOT / "decay_npz"

KERNEL_DIR = CURRENT_ROOT / "kernels"
MATERIALS_DIR = CURRENT_ROOT / "materials"
OUTPUT_DIR = CURRENT_ROOT / "outputs"

KERNEL_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

print("XS_DIR:", XS_DIR)
print("DECAY_DIR:", DECAY_DIR)
print("KERNEL_DIR:", KERNEL_DIR)
print("MATERIALS_DIR:", MATERIALS_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)


In [ ]:
if not XS_DIR.exists():
    raise FileNotFoundError(
        f"Missing neutron data folder: {XS_DIR}\n"
        "Create data/neutron_npz/ and place your processed neutron NPZ files there."
    )

if not DECAY_DIR.exists():
    raise FileNotFoundError(
        f"Missing decay data folder: {DECAY_DIR}\n"
        "Create data/decay_npz/ and place your processed decay NPZ files there."
    )

## 1. User controls

In [ ]:
MATERIAL_NAME = "Steel"          # Change to "Concrete" or another material file.

# Prefer materials/Steel.py, but fall back to Steel.py in the repo root.
MATERIAL_FILE = MATERIALS_DIR / f"{MATERIAL_NAME}.py"
if not MATERIAL_FILE.exists():
    MATERIAL_FILE = CURRENT_ROOT / f"{MATERIAL_NAME}.py"

if not MATERIAL_FILE.exists():
    raise FileNotFoundError(
        f"Missing material file: {MATERIAL_FILE}\n"
        "Place material files in materials/ or in the repo root."
    )

In [ ]:
RANDOM_SEED = None                  # Use None to avoid forcing reproducibility.
RESET_CACHE_BEFORE_RUN = True

In [ ]:
# Transport settings.
TRANSPORT_MODE = "many"          # "single" or "many".
INITIAL_ENERGY_EV = 14.0e6       # 14 MeV source neutron.
BOX_SIZE_M = 5.0
MAX_EVENTS = 1e4                 # Single-source event cap.
MAX_NEUTRONS = 1e3               # Single-source neutron-object cap.
N_SOURCE_NEUTRONS = 1e4          # Used only when TRANSPORT_MODE = "many".
N_WORKERS = 1                    # Many-source worker processes. Use "auto" for all available cores.
MAX_EVENTS_PER_SOURCE = 1e4      # Many-source event cap per source neutron.
MAX_NEUTRONS_PER_SOURCE = 1e3    # Many-source neutron-object cap per source neutron.
MAX_TRAJECTORY_PATHS = 80        # Plot cap used only in many-source mode.

# Transport kernel settings.
USE_TRANSPORT_KERNEL = True
KERNEL_VERSION = "v1"
KERNEL_E_MIN_EV = 1.0e-5
KERNEL_E_MAX_EV = 2.0e7
KERNEL_BINS_PER_DECADE = 100

LOAD_KERNEL_IF_EXISTS = True
BUILD_MF4_ELASTIC_KERNEL = True
MF4_MU_GRID_COUNT = 801


In [ ]:
# Starting neutron state.
START_X = 0.0
START_Y = 0.0
START_DIRECTION = None           # Used only in single-source mode; None samples a random 2D direction.

In [ ]:
# Activity plotting controls.
ACTIVITY_POINTS_PER_DECADE = 100
ACTIVITY_MIN_LINEAR_POINTS = 400
ACTIVITY_HALF_LIVES_TO_SHOW = 8.0
MIN_ACTIVITY_HALF_LIFE_S = 1.0e-12

In [ ]:
print("MATERIAL_NAME:", MATERIAL_NAME)
print("MATERIAL_FILE:", MATERIAL_FILE)
print("TRANSPORT_MODE:", TRANSPORT_MODE)
print("INITIAL_ENERGY_EV:", INITIAL_ENERGY_EV)
print("BOX_SIZE_M:", BOX_SIZE_M)
if str(TRANSPORT_MODE).strip().lower() == "many":
    print("N_SOURCE_NEUTRONS:", N_SOURCE_NEUTRONS)
    print("N_WORKERS:", N_WORKERS)


## 2. Load material

In [ ]:
material, material_df = load_material_summary(
    material_file=str(MATERIAL_FILE),
    xs_dir=str(XS_DIR),
)

display(material_df)

## 3. Inspect reaction probabilities at the initial energy

In [ ]:
reaction_preview_df, open_reactions, Sigma_total = reaction_preview_dataframe(
    material=material,
    energy_eV=INITIAL_ENERGY_EV,
    max_rows=40,
)

display(reaction_preview_df)

In [ ]:
# Build precomputed transport kernel.

if USE_TRANSPORT_KERNEL:
    KERNEL_NPZ_PATH = material_transport_kernel_path(
        kernel_dir=KERNEL_DIR,
        material=material,
        version=KERNEL_VERSION,
    )
    print("KERNEL_NPZ_PATH:", KERNEL_NPZ_PATH)

    if LOAD_KERNEL_IF_EXISTS and KERNEL_NPZ_PATH.exists():
        kernel = load_material_transport_kernel_npz(
            path=KERNEL_NPZ_PATH,
            material=material,
        )
    else:
        kernel = build_material_transport_kernel(
            material=material,
            E_min_eV=KERNEL_E_MIN_EV,
            E_max_eV=KERNEL_E_MAX_EV,
            bins_per_decade=KERNEL_BINS_PER_DECADE,
            print_progress=True,
            build_mf4_elastic=BUILD_MF4_ELASTIC_KERNEL,
            mf4_mu_grid_count=MF4_MU_GRID_COUNT,
        )

        save_material_transport_kernel_npz(
            kernel=kernel,
            path=KERNEL_NPZ_PATH,
        )
        kernel = load_material_transport_kernel_npz(
            path=KERNEL_NPZ_PATH,
            material=material,
        )

    kernel_summary(kernel)

else:
    kernel = None
    print("Transport kernel disabled. Using old runtime reaction-list path.")


## 4. Run event-driven kernel-based transport

In [ ]:
transport_mode = str(TRANSPORT_MODE).strip().lower()
if transport_mode not in {"single", "many"}:
    raise ValueError('TRANSPORT_MODE must be "single" or "many".')

if transport_mode == "single":
    neutrons, hist_df, execution_time = run_single_source_transport(
        material=material,
        initial_energy_eV=INITIAL_ENERGY_EV,
        box_size_m=BOX_SIZE_M,
        start_x=START_X,
        start_y=START_Y,
        start_direction=START_DIRECTION,
        max_events=MAX_EVENTS,
        max_neutrons=MAX_NEUTRONS,
        random_seed=RANDOM_SEED,
        reset_cache=RESET_CACHE_BEFORE_RUN,
        kernel=kernel,
    )
else:
    neutrons, hist_df, execution_time = run_many_source_transport(
        material=material,
        n_source_neutrons=N_SOURCE_NEUTRONS,
        initial_energy_eV=INITIAL_ENERGY_EV,
        box_size_m=BOX_SIZE_M,
        start_x=START_X,
        start_y=START_Y,
        max_events_per_source=MAX_EVENTS_PER_SOURCE,
        max_neutrons_per_source=MAX_NEUTRONS_PER_SOURCE,
        random_seed=RANDOM_SEED,
        reset_cache=RESET_CACHE_BEFORE_RUN,
        kernel=kernel,
        n_workers=N_WORKERS,
    )

display(hist_df)

In [ ]:
if "reaction_sampling_source" in hist_df.columns:
    display(hist_df["reaction_sampling_source"].value_counts(dropna=False))
else:
    print("No reaction_sampling_source column found.")
    print("This usually means Step 2 has not been added to transport.py yet.")

## 5. Transport diagnostics

In [ ]:
include_source_summary = transport_mode == "many"
trajectory_max_paths = None if transport_mode == "single" else MAX_TRAJECTORY_PATHS

time_ordered_events = show_transport_diagnostics(
    hist_df=hist_df,
    material=material,
    include_source_summary=include_source_summary,
)

In [ ]:
plot_neutron_trajectories(
    neutrons=neutrons,
    box_size_m=BOX_SIZE_M,
    max_paths=trajectory_max_paths,
    save_name="neutron_traj.png",
    show_labels=(transport_mode == "single"),
)

## 6. Residual products and activation inventory

In [ ]:
products_df = summarize_residual_products(hist_df)
display(products_df)

activation_df = activation_product_dataframe(hist_df)
display(activation_df)

inventory_summary = activation_inventory_summary(hist_df)
display(inventory_summary)

reaction_inventory = activation_inventory_by_reaction(hist_df)
display(reaction_inventory)

## 7. Decay status

In [ ]:
activation_df_with_decay = add_decay_status_to_activation_products(
    activation_df=activation_df,
    decay_dir=DECAY_DIR,
)

display(activation_df_with_decay)

decay_status_summary = summarize_activation_with_decay_status(
    hist_df=hist_df,
    decay_dir=DECAY_DIR,
)

display(decay_status_summary)

## 8. Direct activity and daughter-chain activity

In [ ]:
time_grid_s, direct_activity_df, chain_activity_df, production_events, activation_df_with_decay = compute_activity_outputs(
    hist_df=hist_df,
    decay_dir=DECAY_DIR,
    points_per_decade=ACTIVITY_POINTS_PER_DECADE,
    min_linear_points=ACTIVITY_MIN_LINEAR_POINTS,
    half_lives_to_show=ACTIVITY_HALF_LIVES_TO_SHOW,
    max_chain_generations=20,
    min_activity_half_life_s=MIN_ACTIVITY_HALF_LIFE_S,
)

In [ ]:
plot_direct_vs_chain_activity_auto(direct_activity_df, chain_activity_df)

plot_total_activity_auto(
    chain_activity_df,
    y_col="total_Ci",
    ylabel="total activity [Ci]",
    save_name="total_activity.png",
)

plot_isotope_activities_auto(chain_activity_df, unit="Bq", save_name="isotope_activities.png")

## 9. Save outputs

In [ ]:
hist_path = OUTPUT_DIR / "transport_history.csv"
products_path = OUTPUT_DIR / "residual_products.csv"
activation_path = OUTPUT_DIR / "activation_products_with_decay.csv"
activity_path = OUTPUT_DIR / "activity_chain.csv"

hist_df.to_csv(hist_path, index=False)
products_df.to_csv(products_path, index=False)
activation_df_with_decay.to_csv(activation_path, index=False)
chain_activity_df.to_csv(activity_path, index=False)

print("Saved:")
print(hist_path)
print(products_path)
print(activation_path)
print(activity_path)